# Embedding Model Attributes

When designing an embedding model, you choose a set of attributes **before training**.
These define the model's capacity — what it can learn, how deeply, and at what cost.

Think of it like building a brain: you decide how many neurons, how many layers of processing,
and how wide the field of vision is. Training then fills that structure with knowledge.

## 1. Dimension — How Many Features

The **dimension** is the length of the output vector. Each dimension captures one abstract feature.

```
embedding("cat") → [0.12, -0.34, 0.56, ...] ← this many numbers
                     dim 0  dim 1  dim 2
```

With few dimensions, the model must compress everything about a word into a tiny space.
With many dimensions, it has room to encode fine-grained distinctions.

| Dimension | Can distinguish | Cannot distinguish |
|---|---|---|
| 2 | animal vs object | cat vs dog |
| 16 | cat vs dog vs bird | Persian cat vs Siamese cat |
| 768 | breeds, behaviors, contexts | very subtle connotations |
| 3072 | almost everything | — |

In [ ]:
import math
import random

random.seed(42)

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(x * x for x in b))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)


# Simulate: with few dimensions, similar things collide
# With many dimensions, they spread out and become distinguishable

words = ["cat", "kitten", "dog", "car", "truck", "banana"]

def make_fake_embeddings(dim, words):
    """Simulate embeddings — similar words get similar vectors."""
    base = {}
    # Create concept clusters with some noise
    concepts = {
        "cat":    [1.0, 0.8, 0.0, -0.5, 0.3, 0.1],
        "kitten": [0.9, 0.9, 0.1, -0.4, 0.4, 0.0],
        "dog":    [0.8, 0.6, 0.3, -0.3, 0.1, 0.5],
        "car":    [-0.5, 0.0, 0.9, 0.8, -0.2, 0.3],
        "truck":  [-0.4, 0.1, 0.8, 0.9, -0.1, 0.4],
        "banana": [0.1, -0.3, -0.2, 0.0, 0.9, -0.6],
    }
    result = {}
    for word in words:
        seed = concepts[word]
        # Stretch or compress to target dimension
        vec = []
        for i in range(dim):
            base_val = seed[i % len(seed)]
            noise = random.gauss(0, 0.3 if dim <= 4 else 0.1)
            vec.append(base_val + noise)
        result[word] = vec
    return result

print("How dimension affects similarity precision")
print("=" * 60)

for dim in [2, 8, 64]:
    random.seed(42)  # same seed for fair comparison
    embs = make_fake_embeddings(dim, words)

    print(f"\n  Dimension = {dim}")
    print(f"  {'Pair':<20} {'Cosine Sim':>10}  Interpretation")
    print(f"  {'-'*55}")

    pairs = [("cat", "kitten"), ("cat", "dog"), ("cat", "car"), ("car", "truck")]
    for w1, w2 in pairs:
        sim = cosine_similarity(embs[w1], embs[w2])
        if sim > 0.8:
            interp = "very similar"
        elif sim > 0.5:
            interp = "somewhat similar"
        elif sim > 0.2:
            interp = "weakly related"
        else:
            interp = "unrelated"
        print(f"  {w1 + ' vs ' + w2:<20} {sim:>10.4f}  {interp}")

print("\nWith dim=2, cat-kitten and cat-dog look equally similar.")
print("With dim=64, the model can clearly rank: kitten > dog > car.")

## 2. Number of Layers — How Deep the Abstraction

Layers control how **sophisticated** the learned features are.

Each layer builds on the previous one, creating increasingly abstract representations:

```
Layer 1:  "cat" → detects letters: c, a, t
Layer 2:  "cat" → detects word identity: this is the word "cat"
Layer 4:  "cat" → detects category: this is an animal
Layer 8:  "cat" → detects connotation: independence, curiosity, agility
Layer 12: "cat" → detects context: "cat" near "debug" means something different
```

A shallow model (2 layers) can only learn surface-level patterns.
A deep model (12+ layers) can learn that "cat" and "independence" are conceptually related,
even though they share no letters and rarely appear next to each other.

In [ ]:
# Simulate: shallow vs deep layers
# Shallow: similarity based on surface co-occurrence
# Deep: similarity based on abstract concepts

# Surface similarity: words that look alike or co-occur directly
surface_scores = {
    ("cat", "cats"):          0.95,  # same word, plural
    ("cat", "kitten"):        0.70,  # co-occur in text
    ("cat", "independence"):  0.05,  # never appear together
    ("cat", "curiosity"):     0.08,  # rarely appear together
    ("bank", "river"):        0.40,  # sometimes co-occur
    ("bank", "money"):        0.40,  # sometimes co-occur
}

# Deep conceptual similarity: understanding meaning
deep_scores = {
    ("cat", "cats"):          0.95,  # still high
    ("cat", "kitten"):        0.90,  # same concept
    ("cat", "independence"):  0.55,  # conceptually linked
    ("cat", "curiosity"):     0.60,  # "curiosity killed the cat" + cat behavior
    ("bank", "river"):        0.30,  # understands these are DIFFERENT "bank"
    ("bank", "money"):        0.75,  # understands financial meaning
}

print("Shallow model (2 layers) vs Deep model (12 layers)")
print("=" * 65)
print(f"  {'Pair':<25} {'Shallow':>8} {'Deep':>8}  Why different?")
print(f"  {'-'*62}")

explanations = {
    ("cat", "cats"):          "both get this right",
    ("cat", "kitten"):        "deep knows they're the same animal",
    ("cat", "independence"):  "deep learned the concept chain",
    ("cat", "curiosity"):     "deep knows cats are curious",
    ("bank", "river"):        "deep disambiguates word senses",
    ("bank", "money"):        "deep picks the dominant meaning",
}

for pair in surface_scores:
    s = surface_scores[pair]
    d = deep_scores[pair]
    label = f"{pair[0]} vs {pair[1]}"
    print(f"  {label:<25} {s:>8.2f} {d:>8.2f}  {explanations[pair]}")

print("\nShallow: only sees co-occurrence. 'cat' and 'independence' = unrelated.")
print("Deep: builds concept chains. cat -> behavior -> aloof -> independence.")

## 3. Attention Heads — How Many Angles at Once

Each attention head looks at the input from a **different perspective** simultaneously.

```
Input: "The bank by the river was steep"

Head 1 (syntax):   "bank" attends to "was" (subject-verb)
Head 2 (topic):    "bank" attends to "river" (what kind of bank?)
Head 3 (adjective): "bank" attends to "steep" (property)
Head 4 (position):  "bank" attends to nearby words
```

More heads = more simultaneous perspectives = richer understanding of each word's role.

| Heads | What it can track |
|---|---|
| 1 | one relationship type at a time |
| 4 | syntax + topic + proximity + meaning |
| 12 | all of above + coreference + negation + style + ... |
| 16 | fine-grained distinctions in all relationship types |

In [ ]:
def softmax(x):
    max_val = max(x)
    exp_vals = [math.exp(v - max_val) for v in x]
    total = sum(exp_vals)
    return [v / total for v in exp_vals]


# Simulate: how attention heads resolve ambiguity
sentence = ["The", "bank", "by", "the", "river", "was", "steep"]

# Single head: one blurred perspective
single_head_scores = {
    "bank": [0.1, 0.0, 0.3, 0.1, 0.4, 0.3, 0.2]  # mixed signal
}

# Multiple heads: each specializes
multi_head_scores = {
    "syntax":    [0.1, 0.0, 0.1, 0.1, 0.1, 0.9, 0.1],  # bank -> was
    "topic":     [0.1, 0.0, 0.2, 0.1, 0.9, 0.1, 0.1],  # bank -> river
    "adjective": [0.1, 0.0, 0.1, 0.1, 0.1, 0.1, 0.9],  # bank -> steep
    "position":  [0.5, 0.0, 0.8, 0.3, 0.2, 0.1, 0.0],  # bank -> nearby
}

print('How attention heads analyze: "The bank by the river was steep"')
print('Target word: "bank"')
print("=" * 65)

print("\n  1 Head (blurred):")
weights = softmax(single_head_scores["bank"])
print(f"  {'Token':<8}", end="")
for tok in sentence:
    print(f"{tok:>8}", end="")
print()
print(f"  {'Weight':<8}", end="")
for w in weights:
    print(f"{w:>8.1%}", end="")
print()
print("  -> Attention is spread out. Can't tell what 'bank' relates to.")

print("\n  4 Heads (specialized):")
for head_name, scores in multi_head_scores.items():
    weights = softmax(scores)
    top_idx = weights.index(max(weights))
    print(f"  {head_name:<10}", end="")
    for i, w in enumerate(weights):
        marker = f"{w:>8.1%}" if i != top_idx else f"  [{w:.0%}]"
        print(f"{marker:>8}", end="")
    print(f"  -> bank-{sentence[top_idx]}")

print("\n  Each head captures a different relationship.")
print("  Combined, the model knows: bank is a riverbank (topic head),")
print("  it 'was' something (syntax head), and it was 'steep' (adj head).")

### Why 768? — Heads × Dims Per Head

768 is not a magic number. It's **12 heads × 64 dimensions per head**.

The total dimension must be evenly divisible by the number of heads,
because each head gets an equal slice of the vector:

```
768-dim vector
|-- Head 1:  dims 0-63    (64 dims)
|-- Head 2:  dims 64-127  (64 dims)
|-- Head 3:  dims 128-191 (64 dims)
|   ...
|-- Head 12: dims 704-767 (64 dims)
```

**64 per head** is the real constant — it was found to be a sweet spot where each head
has enough room to learn a meaningful relationship without wasting compute.

| Model | Heads | Dims/head | Total dimension |
|---|---|---|---|
| BERT-base | 12 | 64 | **768** |
| BERT-large | 16 | 64 | **1024** |
| GPT-2 small | 12 | 64 | **768** |
| GPT-2 medium | 16 | 64 | **1024** |
| GPT-3 | 96 | 128 | **12288** |

### Flat 768 vs 12 Heads × 64

The final vector is the same 768 floats either way. The difference is **how they're computed**.

**Without heads — one big attention (768-dim):**
```
768-dim query x 768-dim key = one similarity score per token
```
One giant dot product. The model asks ONE question about each word pair:
"how related are you?" It blends syntax, topic, meaning all into a single score.

**With 12 heads — 12 separate attentions (64-dim each):**
```
Head 1:  64-dim query x 64-dim key = score (maybe syntax)
Head 2:  64-dim query x 64-dim key = score (maybe topic)
...
Head 12: 64-dim query x 64-dim key = score (maybe coreference)
Then concatenate all 12 outputs -> 768 dims
```

Each head has **its own Q, K, V weight matrices**, so head 1 learns different weights
than head 2. Same total cost (768 × 768 parameters), but heads **force the model
to learn diverse relationships** rather than collapsing everything into one blurry score.

In [ ]:
import random
random.seed(42)

dim_total = 12  # small for demonstration (real: 768)
n_heads = 4     # (real: 12)
dim_per_head = dim_total // n_heads  # 3 (real: 64)

tokens = ["The", "cat", "sat"]

# Each token has a full vector of dim_total
random.seed(42)
vectors = {}
for tok in tokens:
    vectors[tok] = [round(random.gauss(0, 1), 2) for _ in range(dim_total)]

print("Flat 768 vs Multi-Head: Why Heads Matter")
print("=" * 65)
print(f"(Using dim={dim_total}, heads={n_heads}, dim_per_head={dim_per_head} for clarity)\n")

# Show the full vectors
for tok in tokens:
    print(f'  "{tok}" = {vectors[tok]}')

# --- Single head: one big dot product ---
print(f"\n--- Single Head: one {dim_total}-dim dot product ---")
q = vectors["sat"]
k = vectors["cat"]
score = sum(a * b for a, b in zip(q, k))
print(f'  "sat" query x "cat" key = {score:.2f}')
print(f"  One number. Blends ALL relationships into a single score.")

# --- Multi head: split into 4 separate dot products ---
print(f"\n--- {n_heads} Heads: four {dim_per_head}-dim dot products ---")

head_scores = []
for h in range(n_heads):
    start = h * dim_per_head
    end = start + dim_per_head
    q_slice = q[start:end]
    k_slice = k[start:end]
    score = sum(a * b for a, b in zip(q_slice, k_slice))
    head_scores.append(score)
    print(f"  Head {h+1}: dims [{start}:{end}]  {q_slice} x {k_slice} = {score:.2f}")

print(f"\n  4 separate scores: {[round(s, 2) for s in head_scores]}")
print(f"  Each head learned to look at different dims -> different relationship.")
print(f"  Head 1 might track syntax, Head 2 topic, Head 3 proximity, etc.")

print(f"\n--- The key difference ---")
print(f"  Single head:  {sum(head_scores):.2f}  (everything averaged into one number)")
print(f"  Multi-head:   {[round(s, 2) for s in head_scores]}  (4 distinct signals preserved)")
print(f"\n  Same total parameters. But multi-head keeps diverse signals separate,")
print(f"  so the model can reason about multiple relationships at once.")

## 4. Context Window — How Much Text It Sees

The context window is the **maximum number of tokens** the model processes at once.

```
Context = 8:   "The cat sat on the mat and"
               ^^^ only these tokens influence the embedding

Context = 512: The model sees the entire paragraph.
               "In ancient Egypt, cats were worshipped as gods.
                Today, the cat sat on the mat and purred."
               ^^^ "cat" embedding is influenced by "Egypt", "worshipped", "gods"
```

This matters because the **same word means different things** depending on context.

In [ ]:
# Demonstrate: same word, different meaning depending on context window

examples = [
    {
        "word": "apple",
        "short_context": "I ate an apple",
        "long_context": "Steve Jobs founded Apple in a garage. Decades later, I used an Apple to write code.",
        "short_meaning": "fruit",
        "long_meaning": "tech company",
    },
    {
        "word": "python",
        "short_context": "The python was huge",
        "long_context": "After installing pip and setting up a virtual environment, the python script ran perfectly.",
        "short_meaning": "snake (ambiguous)",
        "long_meaning": "programming language",
    },
    {
        "word": "cell",
        "short_context": "Look at this cell",
        "long_context": "Under the microscope, the biologist stained the tissue sample. Look at this cell — the nucleus is clearly visible.",
        "short_meaning": "prison? phone? bio? (ambiguous)",
        "long_meaning": "biological cell",
    },
]

print("Context window determines meaning")
print("=" * 70)

for ex in examples:
    print(f"\n  Word: '{ex['word']}'")
    print(f"  {'-'*65}")
    print(f"  Short context ({len(ex['short_context'].split())} tokens):")
    print(f"    \"{ex['short_context']}\"")
    print(f"    -> Meaning: {ex['short_meaning']}")
    print(f"  Long context ({len(ex['long_context'].split())} tokens):")
    print(f"    \"{ex['long_context']}\"")
    print(f"    -> Meaning: {ex['long_meaning']}")

print("\nShort context window: the model guesses from local words only.")
print("Long context window: the model sees the full picture and disambiguates.")

## 5. Vocabulary Size — How Precisely It Reads Input

Before the model sees any text, a **tokenizer** splits it into tokens.
Vocabulary size determines how many unique tokens the tokenizer knows.

```
Vocab 10k:  "embedding" → ["em", "bed", "ding"]     (3 tokens, choppy)
Vocab 50k:  "embedding" → ["embed", "ding"]          (2 tokens)
Vocab 100k: "embedding" → ["embedding"]               (1 token, clean)
```

Fewer splits = the model sees the word as a whole unit.
More splits = the model must reconstruct meaning from fragments.

In [ ]:
# Simulate tokenization with different vocabulary sizes

def tokenize_small(word):
    """Small vocab: aggressively splits into 2-3 char pieces."""
    tokens = []
    i = 0
    while i < len(word):
        chunk_size = min(3, len(word) - i)
        tokens.append(word[i:i+chunk_size])
        i += chunk_size
    return tokens

def tokenize_medium(word):
    """Medium vocab: splits into known subwords."""
    known = ["transform", "embed", "ding", "er", "back", "propagat", "ion",
             "at", "tent", "tion", "pre", "train", "ing", "un", "super",
             "vised", "self"]
    tokens = []
    remaining = word.lower()
    while remaining:
        matched = False
        for length in range(min(10, len(remaining)), 1, -1):
            if remaining[:length] in known:
                tokens.append(remaining[:length])
                remaining = remaining[length:]
                matched = True
                break
        if not matched:
            tokens.append(remaining[:2])
            remaining = remaining[2:]
    return tokens

def tokenize_large(word):
    """Large vocab: most words are single tokens."""
    known_words = ["transformer", "embedding", "backpropagation", "attention",
                   "pretraining", "unsupervised", "self"]
    if word.lower() in known_words:
        return [word]
    return tokenize_medium(word)


test_words = ["transformer", "embedding", "backpropagation", "attention", "unsupervised"]

print("Tokenization at different vocabulary sizes")
print("=" * 70)
print(f"  {'Word':<20} {'Small (10k)':>20} {'Medium (50k)':>20} {'Large (100k)':>15}")
print(f"  {'-'*70}")

for word in test_words:
    small = tokenize_small(word)
    medium = tokenize_medium(word)
    large = tokenize_large(word)
    print(f"  {word:<20} {str(small):>20} {str(medium):>20} {str(large):>15}")

print(f"\n  {'Tokens per word':>20}", end="")
for label, func in [("Small", tokenize_small), ("Medium", tokenize_medium), ("Large", tokenize_large)]:
    avg = sum(len(func(w)) for w in test_words) / len(test_words)
    print(f"{'avg ' + f'{avg:.1f}':>20}", end="")
print()

print("\nSmall vocab: 'embedding' -> ['emb', 'edd', 'ing'] — model sees fragments.")
print("Large vocab: 'embedding' -> ['embedding'] — model sees the whole word.")
print("\nTradeoff: large vocab = cleaner input but bigger lookup table (more memory).")

## 6. Real Models — What They Chose

Each model makes different tradeoff decisions.

In [ ]:
models = [
    {
        "name": "Word2Vec",
        "year": 2013,
        "dim": 300,
        "layers": 1,
        "heads": 0,
        "context": 5,
        "vocab": "3M",
        "note": "No attention. Sliding window, one layer.",
    },
    {
        "name": "BERT-base",
        "year": 2018,
        "dim": 768,
        "layers": 12,
        "heads": 12,
        "context": 512,
        "vocab": "30k",
        "note": "First transformer encoder for embeddings.",
    },
    {
        "name": "BERT-large",
        "year": 2018,
        "dim": 1024,
        "layers": 24,
        "heads": 16,
        "context": 512,
        "vocab": "30k",
        "note": "Bigger BERT. More layers = deeper abstraction.",
    },
    {
        "name": "OpenAI text-embedding-3-small",
        "year": 2024,
        "dim": 1536,
        "layers": "~24",
        "heads": "~16",
        "context": 8191,
        "vocab": "100k",
        "note": "High dim, huge context. Good general-purpose.",
    },
    {
        "name": "OpenAI text-embedding-3-large",
        "year": 2024,
        "dim": 3072,
        "layers": "~36",
        "heads": "~24",
        "context": 8191,
        "vocab": "100k",
        "note": "Maximum precision. 2x dim = 2x features.",
    },
]

print("Real Embedding Models — Architecture Choices")
print("=" * 80)
print(f"  {'Model':<35} {'Dim':>5} {'Layers':>7} {'Heads':>6} {'Context':>8} {'Vocab':>6}")
print(f"  {'-'*75}")

for m in models:
    print(f"  {m['name']:<35} {m['dim']:>5} {str(m['layers']):>7} {str(m['heads']):>6} {m['context']:>8} {m['vocab']:>6}")

print()
for m in models:
    print(f"  {m['name']}")
    print(f"    {m['note']}")

print("\nTrend: over the years, all attributes have grown.")
print("2013: 300 dim, 1 layer, 5-word window")
print("2024: 3072 dim, 36 layers, 8191-token window")

## 7. Tradeoffs — Why Not Max Everything?

If bigger is better, why not set everything to maximum?

In [ ]:
print("Why not max everything?")
print("=" * 65)

tradeoffs = [
    ("Dimension",
     "More features, finer distinctions",
     "More storage per vector, slower search"),
    ("Layers",
     "Deeper abstraction, conceptual reasoning",
     "Slower inference, needs more training data"),
    ("Attention heads",
     "More relationship types in parallel",
     "More computation per layer"),
    ("Context window",
     "Sees more surrounding text",
     "Memory scales quadratically (O(n^2))"),
    ("Vocab size",
     "Cleaner tokenization",
     "Larger embedding table (vocab x dim matrix)"),
]

for attr, pro, con in tradeoffs:
    print(f"\n  {attr}")
    print(f"    + {pro}")
    print(f"    - {con}")

# Concrete cost example
print("\n" + "=" * 65)
print("Concrete example: storing 1 million document embeddings")
print("-" * 65)

n_docs = 1_000_000
for dim, model_name in [(768, "BERT-base"), (1536, "OpenAI small"), (3072, "OpenAI large")]:
    bytes_per_vec = dim * 4  # float32
    total_gb = (n_docs * bytes_per_vec) / (1024 ** 3)
    print(f"  {model_name:<15} dim={dim:<5} -> {total_gb:.2f} GB for 1M vectors")

print("\nDoubling dimension = doubling storage cost.")
print("For 1 billion documents, this becomes 3 TB vs 12 TB.")

## Summary

All attributes are decided **before training** and cannot be changed after.

| Attribute | What it controls | Low | High |
|---|---|---|---|
| **Dimension** | How many features per vector | blurry, can't distinguish similar things | rich, fine-grained distinctions |
| **Layers** | How deep the abstraction | surface patterns only (co-occurrence) | conceptual chains (cat -> independence) |
| **Attention heads** | How many perspectives at once | narrow focus, misses relationships | multi-angle understanding |
| **Context window** | How much text it sees at once | word-level meaning, ambiguous | document-level, disambiguated |
| **Vocab size** | How precisely it reads input | choppy tokenization, meaning lost | whole-word tokens, clean input |

### The dependency

These attributes are not independent — they must be balanced:

- High dimension + shallow layers = many features but each is superficial
- Deep layers + low dimension = deep reasoning but too few features to express it
- Large context + few heads = sees lots of text but can't track relationships within it

Real models balance all five based on their target use case and compute budget.